# 07 — Model Improvement

## Smart India Real Estate Analytics

### Objective

The objective of this notebook is to investigate whether the current property price prediction model can be improved through better feature representation and model experimentation.

The current selected model is a tuned HistGradientBoosting Regressor with the following test performance:

- MAE: approximately ₹51.82 lakh
- RMSE: approximately ₹1.96 crore
- R²: approximately 0.2175

The existing preprocessing and evaluation notebooks will remain unchanged.

### Improvement Strategy

The first experiment will investigate the representation of `Location`.

The current pipeline groups infrequent locations and applies one-hot encoding. Different minimum-frequency thresholds will be experimentally evaluated to determine whether a more suitable location representation improves model generalization.

The candidate thresholds will be:

- 2 occurrences
- 3 occurrences
- 5 occurrences

All experiments will use the same train/test split and evaluation procedure to ensure a fair comparison.

### Leakage Prevention

Location frequency thresholds will be determined using the training data only.

The test dataset will remain completely unseen during feature-selection decisions and model tuning.

The original raw dataset will not be modified.

## Load Raw Dataset and Reproduce the Baseline Split

The raw dataset is loaded without modification.

The same reproducible train/test split used during the original modeling workflow will be recreated so that the location-frequency experiments can be compared fairly with the existing baseline.

The target variable remains `Price`.

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_PATH = Path("../data/raw/Real Estate Data V21.csv")

df_raw = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df_raw.shape)
print("Target column present:", "Price" in df_raw.columns)

Raw dataset shape: (14528, 9)
Target column present: True


In [3]:
def parse_price(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().replace("₹", "").replace(",", "").lower()

    try:
        if "cr" in value:
            return float(value.replace("cr", "").strip()) * 10_000_000

        elif "lacs" in value:
            return float(value.replace("lacs", "").strip()) * 100_000

        elif "lac" in value:
            return float(value.replace("lac", "").strip()) * 100_000

        elif "k" in value:
            return float(value.replace("k", "").strip()) * 1_000

        elif "l" in value:
            return float(value.replace("l", "").strip()) * 100_000

        else:
            return np.nan

    except ValueError:
        return np.nan


df_processed = df_raw.copy()

df_processed["Price_numeric"] = df_processed["Price"].apply(parse_price)

print(
    f"Valid numeric prices: "
    f"{df_processed['Price_numeric'].notna().sum():,}"
)

print(
    f"Ambiguous/unparsed prices: "
    f"{df_processed['Price_numeric'].isna().sum():,}"
)

Valid numeric prices: 14,525
Ambiguous/unparsed prices: 3


In [4]:
model_data = df_processed.dropna(
    subset=["Price_numeric"]
).copy()

print(f"Modeling dataset shape: {model_data.shape}")
print(
    f"Rows excluded due to ambiguous Price: "
    f"{len(df_processed) - len(model_data):,}"
)

Modeling dataset shape: (14525, 10)
Rows excluded due to ambiguous Price: 3


In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

X = model_data.drop(columns=["Price", "Price_numeric"])
y = model_data["Price_numeric"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
print(f"Training target: {y_train.shape}")
print(f"Test target: {y_test.shape}")

Training features: (11620, 8)
Test features: (2905, 8)
Training target: (11620,)
Test target: (2905,)


In [7]:
title_features = pd.DataFrame(index=model_data.index)

title_text = model_data["Property Title"].astype(str)

# BHK extraction.
# Handles both "2 BHK" and "5+ BHK".
bhk_extract_pattern = r"(?i)(\d+)\s*\+?\s*BHK"

title_features["BHK"] = pd.to_numeric(
    title_text.str.extract(
        bhk_extract_pattern,
        expand=False
    ),
    errors="coerce"
)

# Identify titles explicitly written as "5+ BHK".
five_plus_bhk_pattern = r"(?i)\d+\s*\+\s*BHK"

title_features["BHK_5_PLUS"] = (
    title_text.str.contains(
        five_plus_bhk_pattern,
        regex=True,
        na=False
    )
    .astype(int)
)

# Property configuration category.
title_features["Configuration"] = np.select(
    [
        title_text.str.contains(
            r"(?i)\d+\s*\+?\s*BHK",
            regex=True,
            na=False
        ),
        title_text.str.contains(
            r"(?i)\b\d+\s*RK\b",
            regex=True,
            na=False
        ),
        title_text.str.contains(
            r"(?i)\bStudio\b",
            regex=True,
            na=False
        )
    ],
    [
        "BHK",
        "RK",
        "Studio"
    ],
    default="Unknown"
)

title_features.head()

,BHK,BHK_5_PLUS,Configuration
0,4.0,0,BHK
1,10.0,0,BHK
2,3.0,0,BHK
3,7.0,0,BHK
4,2.0,0,BHK


In [8]:
# Reproduce the original feature-engineering pipeline

X_train_exp_base = X_train.copy()
X_test_exp_base = X_test.copy()

# Add title-derived features
X_train_exp_base["BHK"] = title_features.loc[
    X_train_exp_base.index, "BHK"
]

X_train_exp_base["BHK_5_PLUS"] = title_features.loc[
    X_train_exp_base.index, "BHK_5_PLUS"
]

X_train_exp_base["Configuration"] = title_features.loc[
    X_train_exp_base.index, "Configuration"
]

X_test_exp_base["BHK"] = title_features.loc[
    X_test_exp_base.index, "BHK"
]

X_test_exp_base["BHK_5_PLUS"] = title_features.loc[
    X_test_exp_base.index, "BHK_5_PLUS"
]

X_test_exp_base["Configuration"] = title_features.loc[
    X_test_exp_base.index, "Configuration"
]

print("After engineered features:")
print("Training shape:", X_train_exp_base.shape)
print("Test shape:", X_test_exp_base.shape)

After engineered features:
Training shape: (11620, 11)
Test shape: (2905, 11)


In [9]:
DROP_COLUMNS = [
    "Name",
    "Property Title",
    "Price_per_SQFT",
    "Description"
]

X_train_exp_base = X_train_exp_base.drop(
    columns=DROP_COLUMNS
)

X_test_exp_base = X_test_exp_base.drop(
    columns=DROP_COLUMNS
)

print("After dropping excluded columns:")
print("Training shape:", X_train_exp_base.shape)
print("Test shape:", X_test_exp_base.shape)

print("\nColumns:")
print(X_train_exp_base.columns.tolist())

After dropping excluded columns:
Training shape: (11620, 7)
Test shape: (2905, 7)

Columns:
['Location', 'Total_Area', 'Baths', 'Balcony', 'BHK', 'BHK_5_PLUS', 'Configuration']


In [10]:
X_train_exp_base["City"] = (
    X_train_exp_base["Location"]
    .astype(str)
    .str.split(",")
    .str[-1]
    .str.strip()
)

X_test_exp_base["City"] = (
    X_test_exp_base["Location"]
    .astype(str)
    .str.split(",")
    .str[-1]
    .str.strip()
)

print("Final feature columns:")
print(X_train_exp_base.columns.tolist())

Final feature columns:
['Location', 'Total_Area', 'Baths', 'Balcony', 'BHK', 'BHK_5_PLUS', 'Configuration', 'City']


## Experiment 1 — Location Frequency Threshold

The current preprocessing groups locations occurring fewer than 2 times in the training data into `Other`.

This experiment evaluates whether increasing the minimum location frequency to 3 or 5 improves generalization.

The location frequency is calculated using training data only. The same tuned HistGradientBoosting configuration and the same test set are used for every experiment.

The baseline test R² is 0.2175.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline_exp = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline_exp = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

print("Experimental preprocessing pipelines created successfully.")

Experimental preprocessing pipelines created successfully.


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import gc

# Same features as the original preprocessing pipeline
NUMERIC_FEATURES = [
    "Total_Area",
    "Baths",
    "BHK",
    "BHK_5_PLUS"
]

CATEGORICAL_FEATURES = [
    "Balcony",
    "Configuration",
    "City",
    "Location_Grouped"
]

# Location thresholds to compare
LOCATION_THRESHOLDS = [2, 3, 5]

experiment_results = []

for threshold in LOCATION_THRESHOLDS:

    print("\n" + "=" * 50)
    print(f"Testing location frequency threshold: {threshold}")
    print("=" * 50)

    # Start from the correctly reconstructed baseline features
    X_train_exp = X_train_exp_base.copy()
    X_test_exp = X_test_exp_base.copy()

    # -------------------------------------------------
    # 1. Calculate location frequency from TRAIN only
    # -------------------------------------------------

    location_counts = X_train_exp["Location"].value_counts()

    frequent_locations = set(
        location_counts[
            location_counts >= threshold
        ].index
    )

    print(
        f"Locations retained: {len(frequent_locations):,}"
    )

    # -------------------------------------------------
    # 2. Group rare/unseen locations as "Other"
    # -------------------------------------------------

    X_train_exp["Location_Grouped"] = (
        X_train_exp["Location"].where(
            X_train_exp["Location"].isin(
                frequent_locations
            ),
            "Other"
        )
    )

    X_test_exp["Location_Grouped"] = (
        X_test_exp["Location"].where(
            X_test_exp["Location"].isin(
                frequent_locations
            ),
            "Other"
        )
    )

    # Original Location is replaced by Location_Grouped
    X_train_exp = X_train_exp.drop(
        columns=["Location"]
    )

    X_test_exp = X_test_exp.drop(
        columns=["Location"]
    )

    # -------------------------------------------------
    # 3. Use EXACT baseline preprocessing
    # -------------------------------------------------

    preprocessor_exp = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline_exp,
                NUMERIC_FEATURES
            ),
            (
                "cat",
                categorical_pipeline_exp,
                CATEGORICAL_FEATURES
            )
        ]
    )

    X_train_encoded = preprocessor_exp.fit_transform(
        X_train_exp
    )

    X_test_encoded = preprocessor_exp.transform(
        X_test_exp
    )

    print(
        "Encoded training shape:",
        X_train_encoded.shape
    )

    print(
        "Encoded test shape:",
        X_test_encoded.shape
    )

    # -------------------------------------------------
    # 4. Convert sparse matrix to dense
    #    HistGradientBoosting requires dense input
    # -------------------------------------------------

    X_train_dense = X_train_encoded.toarray().astype(
        np.float32
    )

    X_test_dense = X_test_encoded.toarray().astype(
        np.float32
    )

    # -------------------------------------------------
    # 5. Same tuned HGB configuration as baseline
    # -------------------------------------------------

    model = HistGradientBoostingRegressor(
        min_samples_leaf=50,
        max_leaf_nodes=15,
        max_iter=150,
        learning_rate=0.1,
        l2_regularization=1.0,
        random_state=42
    )

    model.fit(
        X_train_dense,
        y_train
    )

    # -------------------------------------------------
    # 6. Predictions
    # -------------------------------------------------

    predictions = model.predict(
        X_test_dense
    )

    # -------------------------------------------------
    # 7. Evaluation
    # -------------------------------------------------

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    # -------------------------------------------------
    # 8. Store results
    # -------------------------------------------------

    experiment_results.append({
        "Location_Threshold": threshold,
        "Locations_Retained": len(frequent_locations),
        "Features": X_train_dense.shape[1],
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    print(f"MAE:  ₹{mae:,.2f}")
    print(f"RMSE: ₹{rmse:,.2f}")
    print(f"R²:   {r2:.4f}")

    # -------------------------------------------------
    # 9. Free memory
    # -------------------------------------------------

    del (
        X_train_exp,
        X_test_exp,
        X_train_encoded,
        X_test_encoded,
        X_train_dense,
        X_test_dense,
        preprocessor_exp,
        model,
        predictions
    )

    gc.collect()


Testing location frequency threshold: 2
Locations retained: 1,719
Encoded training shape: (11620, 1738)
Encoded test shape: (2905, 1738)
MAE:  ₹5,181,651.46
RMSE: ₹19,559,862.63
R²:   0.2175

Testing location frequency threshold: 3
Locations retained: 1,027
Encoded training shape: (11620, 1046)
Encoded test shape: (2905, 1046)
MAE:  ₹5,169,416.72
RMSE: ₹19,561,088.58
R²:   0.2174

Testing location frequency threshold: 5
Locations retained: 496
Encoded training shape: (11620, 515)
Encoded test shape: (2905, 515)
MAE:  ₹5,189,865.25
RMSE: ₹19,536,092.27
R²:   0.2194
